# MotionHiFlow — Benchmark Export (Colab)

This notebook generates MotionHiFlow motions for the shared T2M benchmark and packages them
for the common benchmark notebook (`Benchmark_simple_version_English.ipynb`, STEP 3.5).

**Output:** `MotionHiFlow_standardized_<split>.zip` containing

```
MotionHiFlow/<prompt_id>.npy    standardised [T, 22, 3], 20 fps, metres,
                                +X = Right, +Y = Up, +Z = Forward,
                                ground Y = 0, initial root XZ = (0, 0)
MotionHiFlow/<prompt_id>.json   per-prompt metadata (seed, frames, facing check)
MotionHiFlow/manifest.json      run-level manifest
```

plus `MotionHiFlow_gifs_<split>.zip` — one GIF per prompt (original MotionHiFlow-pipeline renderer),
for checking the motions and for Human Gold labelling.

## How to use

1. **Runtime → Change runtime type → GPU.**
2. Run the cells from top to bottom. No runtime restart is needed.
3. Step 4 asks you to upload the benchmark definition JSON (v1.2 works; lengths follow the team rule
   Easy 100 / Medium 150 / Hard 196 frames — MotionHiFlow generates 150 as 148).
4. Step 6 is a short left/right sanity check on C2-01 / C2-02 — look at it before Step 7.
5. Step 8 downloads the motion ZIP and the GIF ZIP. Upload the motion ZIP to the benchmark notebook.

The runner code is embedded in Step 2, so no other file is needed.

## 1. Settings and GPU check

In [ ]:
# ------------------------------------------------------------------
# Settings — edit here only.
# ------------------------------------------------------------------
SPLIT = "Pilot"          # "Pilot" or "Main"; None = every prompt in the JSON
SEED = 1
BATCH_SIZE = 16          # lower it (e.g. 8) if the GPU runs out of memory
CHUNK_SIZE = 200         # prompts per model load; also the resume granularity
MAKE_GIFS = True         # one GIF per prompt, ~15–30 s each (Pilot ≈ 6–8 min); False to skip

# Save to Google Drive so an interrupted Main run can resume in a new session.
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/motionhiflow_benchmark_exports"

import shutil, subprocess, sys
print("Python:", sys.version.split()[0])
gpu_name = ""
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    gpu_name = gpu.stdout.strip() if gpu.returncode == 0 else ""
if gpu_name:
    print("GPU   :", gpu_name)
else:
    print("WARNING: no GPU detected. Runtime → Change runtime type → GPU.")

## 2. Write the runner module

This cell writes `motionhiflow_benchmark_runner.py` (v5.1) to `/content`. Do not edit it here;
change the settings in Step 1 instead.

In [ ]:
%%writefile /content/motionhiflow_benchmark_runner.py
"""
MotionHiFlow benchmark runner V5
================================

Self-contained runner with the same external setup pattern used by the
MoMADiff benchmark runner:

    runner = get_motionhiflow_runner(
        setup=True,
        install_dependencies=True,
        download_checkpoints=True,
        seed=1,
    )

    # Single prompt (debugging / quick tests; reloads the model on every call):
    motion = runner.generate("A person walks forward.", "C1-01", motion_length=100)

    # Whole benchmark split (batched; the model is loaded once per chunk):
    report = runner.export_benchmark_package(
        "/content/benchmark_definition_v1.0.json",
        split="Pilot",
    )
    # -> /content/benchmark_exports/MotionHiFlow_standardized_pilot.zip
    #    containing MotionHiFlow/<prompt_id>.npy (+ .json) and manifest.json,
    #    the layout the common benchmark notebook expects in STEP 3.5.

Output:
    numpy.ndarray [T, 22, 3], benchmark-standard coordinates:
        20 fps, metres, +X = Right, +Y = Up, +Z = Forward,
        ground Y = 0, initial root XZ = (0, 0).

The setup logic is based on the MotionHiFlow Colab notebook. The standardisation
is ported verbatim from `benchmark_utils.standardise_motionhiflow`, so this runner
produces the same joints as the notebook pipeline without importing that module.

Changes from V4:
  - generate() now applies the full benchmark standardisation (ground, origin,
    X mirroring); V4 only validated shape and finiteness.
  - seed is passed to run.sh (default 1, same as the notebook).
  - motion_length must be given explicitly and, by default, must be one of the
    official benchmark lengths (now 100 / 150 / 196, see V5.2 below).
  - every generate() call writes into its own output directory, so a stale file
    from an earlier run can never be picked up.
  - the generated frame count is checked against the requested length.
  - the prompt is passed via a prompt file (text_path) instead of
    text_prompt=..., because Hydra rejects prompts containing commas,
    parentheses or apostrophes on the command line.
  - export_benchmark_package(): batched generation of a whole benchmark split,
    written as <model>/<prompt_id>.npy and zipped for the benchmark notebook.

Changes in V5.2:
  - supports definition v1.2, which has no target_frames_20fps: lengths follow
    the team rule by difficulty (Easy 100, Medium 150, Hard 196);
    target_frames_20fps is still used when a definition provides it.
  - MotionHiFlow generates multiples of 4 frames, so 150 is produced as 148;
    requested and effective lengths are both recorded.

Changes in V5.3:
  - export_benchmark_package(make_gifs=True) renders every standardised motion
    with the original render_standard_gif (MotionHiFlow pipeline, unchanged) and
    zips them separately as <model>_gifs_<split>.zip.
"""

from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import zipfile
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional


RUNNER_VERSION = "5.3-gifs"
REPO_URL = "https://github.com/ai-lh/MotionHiFlow.git"

# Team length rule (20 fps). Definition v1.2 no longer prescribes lengths
# ("generation length is determined by each source model"); for MotionHiFlow
# the team uses one length per difficulty level.
DEFAULT_LENGTH_BY_DIFFICULTY = {"Easy": 100, "Medium": 150, "Hard": 196}

# Requested lengths generate() accepts by default.
BENCHMARK_LENGTHS = tuple(sorted(set(DEFAULT_LENGTH_BY_DIFFICULTY.values())))

# Name of the experiment sub-directory MotionHiFlow writes into.
EXPERIMENT_NAME = "t2m_tmdit_16d"

# MotionHiFlow rounds requested lengths down to a multiple of this value.
LENGTH_MULTIPLE = 4

N_JOINTS = 22


# ======================================================================
# Standardisation (ported from benchmark_utils.py — keep in sync)
# ======================================================================

def standardise_motionhiflow(motion, mirror_x: bool = True, n_joints: int = N_JOINTS):
    """
    Standardise one motion clip. Identical to benchmark_utils.standardise_motionhiflow.

    1. Ground alignment  — lift the lowest point of the whole clip to Y = 0.
    2. Origin alignment  — move the first-frame root XZ to (0, 0).
    3. Axis convention   — mirror X so that +X = Right (only when mirror_x is True).

    Final convention: +X = Right, -X = Left, +Z = Forward, -Z = Backward, +Y = Up.

    Returns (motion_std, metadata).
    """
    import numpy as np

    motion = np.asarray(motion, dtype=np.float32)

    if motion.ndim != 3:
        raise ValueError(f"Expected [T, J, 3], got {motion.shape}")
    if motion.shape[1] != n_joints:
        raise ValueError(f"Expected {n_joints} joints, got {motion.shape[1]}")
    if motion.shape[2] != 3:
        raise ValueError(f"Expected XYZ, got {motion.shape}")
    if motion.shape[0] < 1:
        raise ValueError("Motion has no frames.")
    if not np.isfinite(motion).all():
        raise ValueError("Motion contains NaN or Inf.")

    motion_std = motion.copy()

    # 1. Ground alignment: global minimum, so jumps/crouches do not sink the body.
    original_floor = float(motion_std[:, :, 1].min())
    motion_std[:, :, 1] -= original_floor

    # 2. Origin alignment (joint 0 is the root / pelvis).
    original_root_x = float(motion_std[0, 0, 0])
    original_root_z = float(motion_std[0, 0, 2])
    motion_std[:, :, 0] -= original_root_x
    motion_std[:, :, 2] -= original_root_z

    # 3. Axis convention: raw HumanML3D +X points to the character's left.
    if mirror_x:
        motion_std[:, :, 0] *= -1.0

    metadata = {
        "Original_Floor_Y": original_floor,
        "Original_Root_X": original_root_x,
        "Original_Root_Z": original_root_z,
        "New_Floor_Y": float(motion_std[:, :, 1].min()),
        "New_Root_X": float(motion_std[0, 0, 0]),
        "New_Root_Z": float(motion_std[0, 0, 2]),
        "Mirrored_X": bool(mirror_x),
        "X_Convention": "+X=Right, -X=Left",
        "Z_Convention": "+Z=Forward, -Z=Backward",
        "Y_Convention": "+Y=Up",
    }
    return motion_std, metadata


def estimate_hml3d_initial_forward(motion, n_frames: int = 5):
    """
    Estimate the initial facing direction. Identical to
    benchmark_utils.estimate_hml3d_initial_forward. Call on the RAW motion.

    Returns (forward_vector | None, angle_in_degrees_relative_to_+Z | None).
    """
    import numpy as np

    motion = np.asarray(motion, dtype=np.float32)
    n = min(n_frames, len(motion))

    R_HIP, L_HIP = 2, 1
    R_SHOULDER, L_SHOULDER = 17, 16

    across_hip = motion[:n, R_HIP] - motion[:n, L_HIP]
    across_shoulder = motion[:n, R_SHOULDER] - motion[:n, L_SHOULDER]

    across = (across_hip + across_shoulder).mean(axis=0)
    across[1] = 0.0

    norm = np.linalg.norm(across)
    if norm < 1e-8:
        return None, None
    across /= norm

    up = np.array([0.0, 1.0, 0.0], dtype=np.float32)
    forward = np.cross(up, across)
    forward[1] = 0.0
    forward /= (np.linalg.norm(forward) + 1e-8)

    angle_deg = float(np.degrees(np.arctan2(forward[0], forward[2])))
    return forward, angle_deg


def check_left_right_convention(motion_std, expect: str = "left",
                                min_displacement: float = 0.3):
    """
    Verify the left/right convention on a STANDARDISED clip whose prompt names a
    direction. Identical to benchmark_utils.check_left_right_convention.

    Returns (ok | None, dx). ok is None when the root barely moves sideways.
    """
    import numpy as np

    if expect not in ("left", "right"):
        raise ValueError('expect must be "left" or "right"')

    root = np.asarray(motion_std)[:, 0, :]
    dx = float(root[-1, 0] - root[0, 0])

    if abs(dx) < min_displacement:
        return None, dx

    moved = "right" if dx > 0 else "left"
    return moved == expect, dx


# ======================================================================
# GIF rendering (ported verbatim from benchmark_utils.render_standard_gif;
# only a lazy `import numpy as np` was added to render_standard_gif)
# ======================================================================

# Camera angle for the rendered GIFs. elev=30 looks down at the scene; a negative elevation
# views it from below, which makes left and right easy to misjudge by eye.
DEFAULT_ELEV = 30
DEFAULT_AZIM = -45


# Kinematic chains of the HumanML3D 22-joint skeleton.
HML3D_KINEMATIC_CHAIN = [
    [0, 2, 5, 8, 11],      # right leg
    [0, 1, 4, 7, 10],      # left leg
    [0, 3, 6, 9, 12, 15],  # spine and head
    [9, 14, 17, 19, 21],   # right arm
    [9, 13, 16, 18, 20],   # left arm
]


def get_scene_limits(motion, margin: float = 1.0, min_span: float = 3.0):
    """
    Derive the scene extent from the root trajectory.

    margin   : padding around the trajectory, in metres
    min_span : minimum visible span, so in-place motions still get enough room
    """

    root = motion[:, 0, :]

    x_min = float(root[:, 0].min()) - margin
    x_max = float(root[:, 0].max()) + margin
    z_min = float(root[:, 2].min()) - margin
    z_max = float(root[:, 2].max()) + margin

    # Widen the view for small-displacement motions such as an in-place jump.
    if x_max - x_min < min_span:
        centre = (x_min + x_max) / 2
        x_min, x_max = centre - min_span / 2, centre + min_span / 2
    if z_max - z_min < min_span:
        centre = (z_min + z_max) / 2
        z_min, z_max = centre - min_span / 2, centre + min_span / 2

    # The vertical range comes from the whole skeleton, since raised hands exceed head height.
    y_min = 0.0
    y_max = float(motion[:, :, 1].max()) + 0.25

    return x_min, x_max, z_min, z_max, y_min, y_max


def render_standard_gif(motion, output_file, prompt_id: str, prompt: str,
                        fps: int = 20, elev: float = DEFAULT_ELEV,
                        azim: float = DEFAULT_AZIM, verbose: bool = True) -> Path:
    """
    Render one standardised motion clip to a GIF, with the axis directions drawn in.

    Note the axis mapping: matplotlib's 3D axes do not map one-to-one onto the motion axes.
    The motion's Y (height) is plotted on matplotlib's Z axis (vertical).

    The axis arrows and labels live in data space, so they rotate with the camera and stay
    truthful at any (elev, azim) — only the viewpoint changes, never the data.

    matplotlib is imported here rather than at module level so that importing this module
    never depends on a display backend being configured.
    """
    import numpy as np  # lazy, like the rest of this runner

    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation, PillowWriter

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    x_min, x_max, z_min, z_max, y_min, y_max = get_scene_limits(motion)

    if verbose:
        print(f"{prompt_id}: X=[{x_min:.2f}, {x_max:.2f}], "
              f"Z=[{z_min:.2f}, {z_max:.2f}], Y=[{y_min:.2f}, {y_max:.2f}]")

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection="3d")

    # Compute the ground grid once, outside the frame loop.
    GX, GZ = np.meshgrid(np.linspace(x_min, x_max, 10), np.linspace(z_min, z_max, 10))
    GY = np.zeros_like(GX)

    L = 0.8  # length of the axis arrows
    AXES = [
        (( L, 0, 0), "+X RIGHT"),
        ((-L, 0, 0), "-X LEFT"),
        ((0,  L, 0), "+Z FORWARD"),
        ((0, -L, 0), "-Z BACKWARD"),
        ((0, 0,  L), "+Y UP"),
    ]

    def update(frame):
        ax.cla()
        joints = motion[frame]

        # Stick figure. Note the argument order: motion X -> plot X, motion Z -> plot Y,
        # motion Y -> plot Z.
        for chain in HML3D_KINEMATIC_CHAIN:
            p = joints[chain]
            ax.plot(p[:, 0], p[:, 2], p[:, 1], linewidth=2)
        ax.scatter(joints[:, 0], joints[:, 2], joints[:, 1], s=10)

        # Root trajectory projected onto the ground.
        root = motion[:frame + 1, 0, :]
        ax.plot(root[:, 0], root[:, 2], np.zeros(len(root)), linestyle="--", linewidth=1.5)

        ax.plot_wireframe(GX, GZ, GY, linewidth=0.3, alpha=0.35)

        for (dx, dy, dz), label in AXES:
            ax.quiver(0, 0, 0, dx, dy, dz, arrow_length_ratio=0.12)
            ax.text(dx, dy, dz, label)

        # Fixed extent and camera, so clips stay visually comparable.
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(z_min, z_max)
        ax.set_zlim(y_min, y_max)
        ax.set_box_aspect((1, 1, 0.7))
        ax.view_init(elev=elev, azim=azim)

        ax.set_xlabel("X (+Right / -Left)")
        ax.set_ylabel("Z (+Forward / -Backward)")
        ax.set_zlabel("Y (+Up)")
        ax.grid(True)
        ax.set_title(f"{prompt_id}: {prompt}\nFrame {frame + 1}/{len(motion)}")

    animation = FuncAnimation(fig, update, frames=len(motion), interval=1000 / fps)
    animation.save(str(output_file), writer=PillowWriter(fps=fps), dpi=100)
    plt.close(fig)  # must close, otherwise batch rendering leaks memory

    if verbose:
        print("Saved:", output_file)
    return output_file


# ======================================================================
# Runner
# ======================================================================

class MotionHiFlowRunner:
    def __init__(
        self,
        repo_dir: str = "/content/MotionHiFlow",
        gpu_id: int = 0,
        seed: int = 1,
        default_motion_length: Optional[int] = None,
        allowed_lengths: Optional[Iterable[int]] = BENCHMARK_LENGTHS,
        mirror_x: bool = True,
        facing_tolerance_deg: float = 15.0,
        length_by_difficulty: Optional[dict] = None,
    ):
        """
        Args:
            seed                  : passed to run.sh; the notebook default is 1.
            default_motion_length : optional fallback when generate() gets no
                                    motion_length. None (default) makes the length
                                    mandatory, which is what the benchmark needs.
            length_by_difficulty  : length rule used by export_benchmark_package()
                                    when a prompt has no target_frames_20fps.
                                    Default {"Easy": 100, "Medium": 150, "Hard": 196}.
            allowed_lengths       : lengths generate() accepts. Defaults to the
                                    official benchmark lengths; pass None to allow
                                    any multiple of 4 (e.g. for quick tests).
            mirror_x              : True for raw MotionHiFlow output (+X = Left).
                                    Do not change without re-checking C2-01/C2-02.
        """
        self.repo_dir = Path(repo_dir)
        self.gpu_id = int(gpu_id)
        self.seed = int(seed)
        self.allowed_lengths = (
            None if allowed_lengths is None
            else tuple(int(x) for x in allowed_lengths)
        )
        self.mirror_x = bool(mirror_x)
        self.facing_tolerance_deg = float(facing_tolerance_deg)
        self.length_by_difficulty = dict(length_by_difficulty or DEFAULT_LENGTH_BY_DIFFICULTY)

        self.default_motion_length = None
        if default_motion_length is not None:
            self.default_motion_length = self._validate_length(default_motion_length)

        # Metadata of the most recent generate() call (paths, seed, standardisation).
        self.last_metadata: Optional[dict] = None

    # ------------------------------------------------------------------
    # Setup
    # ------------------------------------------------------------------

    def _run(self, cmd, cwd=None, env=None):
        """Run a command and fail loudly if it fails."""
        print("[MotionHiFlow] $", " ".join(map(str, cmd)))

        # MotionHiFlow is a PyTorch pipeline.  On current Colab images,
        # Transformers may otherwise auto-detect TensorFlow/JAX and fail
        # inside that unrelated stack before PreTrainedModel is exposed.
        run_env = os.environ.copy()
        if env:
            run_env.update(env)
        run_env["USE_TF"] = "0"
        run_env["USE_FLAX"] = "0"

        subprocess.run(
            [str(x) for x in cmd],
            cwd=str(cwd) if cwd else None,
            env=run_env,
            check=True,
        )

    def _pip(self, *packages):
        self._run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *packages]
        )

    def clone_repository(self):
        run_sh = self.repo_dir / "run.sh"

        if run_sh.exists():
            print("[MotionHiFlow] Repository already exists — skip clone.")
            return

        if self.repo_dir.exists():
            print("[MotionHiFlow] Removing incomplete repository.")
            shutil.rmtree(self.repo_dir)

        self.repo_dir.parent.mkdir(parents=True, exist_ok=True)

        self._run(
            ["git", "clone", REPO_URL, str(self.repo_dir)]
        )

        required = [
            "run.sh",
            "gen_t2m.py",
            "prepare.sh",
            "requirements.txt",
            "configs",
            "src",
        ]

        missing = [
            x for x in required
            if not (self.repo_dir / x).exists()
        ]

        if missing:
            raise RuntimeError(
                "MotionHiFlow repository is incomplete. Missing: "
                + ", ".join(missing)
            )

        print("[MotionHiFlow] Repository ready.")

    def install_project_dependencies(self):
        """
        Install MotionHiFlow dependencies.

        The reproduction notebook pinned NumPy 1.26.4 and SciPy 1.11.1.
        Those releases do not support Python 3.13. Therefore:
          - Python <= 3.12: preserve the verified notebook versions.
          - Python >= 3.13: use a compatible NumPy/SciPy pair while
            preserving the verified MotionHiFlow transformer stack.

        Generation is executed in a fresh subprocess, so it sees the
        installed package versions without relying on already-imported
        modules in this notebook process.
        """
        marker = self.repo_dir / ".benchmark_dependencies_ready_v2"

        if marker.exists():
            print("[MotionHiFlow] Dependencies already prepared — skip.")
            return

        py_major, py_minor = sys.version_info[:2]
        print(f"[MotionHiFlow] Host Python: {py_major}.{py_minor}")

        print("[MotionHiFlow] Installing setup utilities...")
        self._pip("gdown", "huggingface_hub==0.34.4")

        if (py_major, py_minor) >= (3, 13):
            numpy_spec = "numpy>=2.1,<2.3"
            scipy_spec = "scipy>=1.14,<1.17"
            print(
                "[MotionHiFlow] Python 3.13+ detected; "
                "using compatible NumPy/SciPy versions."
            )
        else:
            numpy_spec = "numpy==1.26.4"
            scipy_spec = "scipy==1.11.1"
            print(
                "[MotionHiFlow] Python <=3.12 detected; "
                "using notebook-verified NumPy/SciPy versions."
            )

        self._pip(numpy_spec, scipy_spec)

        requirements = self.repo_dir / "requirements.txt"
        filtered = self.repo_dir / "requirements_benchmark_colab.txt"

        excluded_prefixes = (
            "--extra-index-url",
            "torch==",
            "torchvision==",
            "numpy==",
            "scipy==",
            "transformers",
            "peft",
            "tokenizers",
            "accelerate",
            "huggingface-hub==",
            "diffusers==",
        )

        kept = []
        for line in requirements.read_text(encoding="utf-8").splitlines():
            stripped = line.strip()
            if not stripped:
                continue
            if any(stripped.startswith(x) for x in excluded_prefixes):
                continue
            kept.append(line)

        filtered.write_text("\n".join(kept) + "\n", encoding="utf-8")

        print("[MotionHiFlow] Installing remaining project requirements...")
        self._run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)]
        )

        print("[MotionHiFlow] Installing verified transformer stack...")
        self._pip(
            "transformers==4.53.2",
            "peft==0.17.1",
            "tokenizers==0.21.4",
            "accelerate==1.10.1",
            "huggingface-hub==0.34.4",
            "diffusers==0.35.1",
        )

        # Verify in a fresh interpreter, not this already-running notebook.
        verify_code = r"""
import sys
import numpy
import scipy
import torch
import transformers
import peft
import tokenizers
import accelerate
import huggingface_hub
import diffusers
from transformers import HybridCache, PreTrainedModel

print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Accelerate:", accelerate.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Diffusers:", diffusers.__version__)
print("CUDA:", torch.cuda.is_available())
print("PreTrainedModel: OK")
print("PEFT import: OK")

assert huggingface_hub.__version__ == "0.34.4"
"""
        print("[MotionHiFlow] Verifying imports in a fresh interpreter...")
        self._run([sys.executable, "-c", verify_code])

        marker.write_text(
            f"python={py_major}.{py_minor}\n"
            f"numpy={numpy_spec}\n"
            f"scipy={scipy_spec}\n",
            encoding="utf-8",
        )
        print("[MotionHiFlow] Dependencies ready.")

    def apply_compatibility_patches(self):
        """Apply the NumPy visualization patches from the notebook."""
        patches = {
            "src/visualization/common/quaternion.py": [
                ("np.finfo(np.float).eps", "np.finfo(float).eps"),
            ],
            "src/visualization/AnimationStructure.py": [
                (".astype(np.int)", ".astype(int)"),
            ],
            "src/visualization/remove_fs.py": [
                (".astype(np.float)", ".astype(float)"),
            ],
        }

        for rel, replacements in patches.items():
            path = self.repo_dir / rel
            if not path.exists():
                continue

            text = path.read_text(encoding="utf-8")
            for old, new in replacements:
                text = text.replace(old, new)
            path.write_text(text, encoding="utf-8")

        animation = self.repo_dir / "src/visualization/Animation.py"
        if animation.exists():
            text = animation.read_text(encoding="utf-8")
            text = text.replace(
                "import numpy.core.umath_tests as ut\n",
                "",
            )
            text = text.replace(
                "ut.matrix_multiply(",
                "np.matmul(",
            )
            animation.write_text(text, encoding="utf-8")

        print("[MotionHiFlow] Compatibility patches applied.")

    # ------------------------------------------------------------------
    # Resources / checkpoints
    # ------------------------------------------------------------------

    def required_checkpoint_files(self):
        return [
            self.repo_dir
            / "logs/t2m_tmdit_16d/checkpoints/net_best_fid.tar",

            self.repo_dir
            / "logs/t2m_vae_agcn_16d/checkpoints/net_best_fid.tar",
        ]

    def verify_checkpoints(self, raise_on_missing=True):
        missing = [
            p for p in self.required_checkpoint_files()
            if not p.exists()
        ]

        if missing and raise_on_missing:
            raise FileNotFoundError(
                "Missing MotionHiFlow checkpoints:\n"
                + "\n".join(f" - {p}" for p in missing)
            )

        if not missing:
            print("[MotionHiFlow] Required checkpoints verified.")

        return len(missing) == 0

    def download_pretrained_checkpoints(self):
        if self.verify_checkpoints(raise_on_missing=False):
            print("[MotionHiFlow] Checkpoints already exist — skip download.")
            return

        print("[MotionHiFlow] Downloading pretrained checkpoints...")
        self._run(
            ["bash", "prepare.sh", "pretrained"],
            cwd=self.repo_dir,
        )

        self.verify_checkpoints(raise_on_missing=True)

    def prepare_generation_resources(self):
        """
        Prepare lightweight resources used by the HumanML3D/T2M config.

        The notebook prepared evaluator/GloVe/CLIP and HumanML3D resources
        before generation. Here we avoid downloading the full HumanML3D
        motion corpus; only Mean.npy / Std.npy are requested.
        """

        if importlib.util.find_spec("gdown") is None:
            self._pip("gdown")

        # Local CLIP text encoder required by MotionHiFlow generation.
        clip_dir = self.repo_dir / "deps" / "clip-vit-base-patch32"
        clip_config = clip_dir / "config.json"
        clip_weights_bin = clip_dir / "pytorch_model.bin"
        clip_weights_safe = clip_dir / "model.safetensors"

        if (
            clip_config.exists()
            and (clip_weights_bin.exists() or clip_weights_safe.exists())
        ):
            print("[MotionHiFlow] CLIP model already exists — skip download.")
        else:
            print("[MotionHiFlow] Preparing local CLIP model...")
            clip_dir.mkdir(parents=True, exist_ok=True)

            script = (
                "from huggingface_hub import snapshot_download;"
                "snapshot_download("
                "repo_id='openai/clip-vit-base-patch32',"
                f"local_dir={str(clip_dir)!r},"
                "allow_patterns=["
                "'config.json',"
                "'preprocessor_config.json',"
                "'tokenizer.json',"
                "'tokenizer_config.json',"
                "'special_tokens_map.json',"
                "'merges.txt',"
                "'vocab.json',"
                "'pytorch_model.bin',"
                "'model.safetensors'"
                "]"
                ")"
            )
            self._run([sys.executable, "-c", script], cwd=self.repo_dir)

            if not clip_config.exists() or not (
                clip_weights_bin.exists() or clip_weights_safe.exists()
            ):
                raise FileNotFoundError(
                    "MotionHiFlow CLIP download did not produce the required "
                    f"local model files in {clip_dir}"
                )

        print("[MotionHiFlow] CLIP model ready.")

        # HumanML3D evaluator resources.
        eval_marker = (
            self.repo_dir
            / "deps/evaluators/t2m/Comp_v6_KLD005/meta/mean.npy"
        )

        if not eval_marker.exists():
            print("[MotionHiFlow] Preparing HumanML3D evaluator resources...")
            dest = self.repo_dir / "deps/evaluators/t2m"
            dest.mkdir(parents=True, exist_ok=True)

            zip_path = dest / "humanml3d_evaluator.zip"

            self._run([
                sys.executable,
                "-m",
                "gdown",
                "19C_eiEr0kMGlYVJy_yFL6_Dhk3RvmwhM",
                "-O",
                str(zip_path),
            ])

            self._run([
                "unzip", "-q", "-o",
                str(zip_path),
                "-d", str(dest),
            ])

        # GloVe resources used by MotionHiFlow configs/evaluation utilities.
        glove_marker = self.repo_dir / "glove/our_vab_data.npy"

        if not glove_marker.exists():
            print("[MotionHiFlow] Preparing GloVe resources...")
            zip_path = self.repo_dir / "glove_data.zip"

            self._run([
                sys.executable,
                "-m",
                "gdown",
                "1cmXKUT31pqd7_XpJAiWEo1K81TMYHA5n",
                "-O",
                str(zip_path),
            ])

            self._run([
                "unzip", "-q", "-o",
                str(zip_path),
                "-d", str(self.repo_dir),
            ])

        # Keep the deps/glove mirror used in the notebook.
        glove_src = self.repo_dir / "glove"
        glove_dst = self.repo_dir / "deps/glove"

        if glove_src.exists() and not glove_dst.exists():
            shutil.copytree(glove_src, glove_dst)

        # Download only HumanML3D normalization files, not the full dataset.
        data_root = self.repo_dir / "datasets/humanml3d"
        mean_file = data_root / "Mean.npy"
        std_file = data_root / "Std.npy"

        if not (mean_file.exists() and std_file.exists()):
            print("[MotionHiFlow] Preparing HumanML3D normalization files...")

            script = (
                "from huggingface_hub import snapshot_download;"
                f"snapshot_download(repo_id='NamYeongCho/HumanML3D_new',"
                f"repo_type='dataset',local_dir={str(data_root)!r},"
                "allow_patterns=['Mean.npy','Std.npy'])"
            )

            self._run([sys.executable, "-c", script])

        print("[MotionHiFlow] Generation resources ready.")

    def setup_environment(
        self,
        install_dependencies=True,
        download_checkpoints=True,
    ):
        print(f"[MotionHiFlow] Runner {RUNNER_VERSION}")
        print("[MotionHiFlow] Setting up environment...")

        self.clone_repository()

        if install_dependencies:
            self.install_project_dependencies()

        self.apply_compatibility_patches()

        if download_checkpoints:
            self.download_pretrained_checkpoints()

        self.prepare_generation_resources()

        self.verify_checkpoints()

        print("[MotionHiFlow] Environment ready.")

    # ------------------------------------------------------------------
    # Model / generation API
    # ------------------------------------------------------------------

    def load_model(self):
        """
        MotionHiFlow's official gen_t2m.py loads the Flow model and VAE for
        each generation process. This method therefore verifies readiness
        rather than keeping a second in-notebook model copy in memory.
        """
        self.verify_checkpoints()

        if not (self.repo_dir / "gen_t2m.py").exists():
            raise FileNotFoundError(
                f"Missing {self.repo_dir / 'gen_t2m.py'}"
            )

        print(
            "[MotionHiFlow] Model files ready. "
            "Models will be loaded by gen_t2m.py during generate()."
        )
        return self

    def _validate_length(self, length) -> int:
        length = int(length)

        if length < LENGTH_MULTIPLE:
            raise ValueError(f"motion_length must be >= {LENGTH_MULTIPLE}.")

        if self.allowed_lengths is not None and length not in self.allowed_lengths:
            raise ValueError(
                f"motion_length={length} is not an official benchmark length "
                f"{self.allowed_lengths}. Pass allowed_lengths=None to the runner "
                "for non-benchmark tests."
            )

        return length

    @staticmethod
    def effective_length(length) -> int:
        """
        Frames MotionHiFlow actually produces for a requested length: gen_t2m.py
        works in 4-frame tokens and rounds down (150 -> 148, 100 -> 100, 196 -> 196).
        """
        return (int(length) // LENGTH_MULTIPLE) * LENGTH_MULTIPLE

    def length_for_prompt(self, prompt: dict):
        """
        Requested length and its source for one benchmark prompt:
        target_frames_20fps if the definition gives it (v1.0/1.1), otherwise the
        team rule by difficulty (v1.2).
        """
        if prompt.get("target_frames_20fps") is not None:
            return int(prompt["target_frames_20fps"]), "target_frames_20fps"
        difficulty = prompt.get("difficulty")
        if difficulty not in self.length_by_difficulty:
            raise KeyError(
                f"{prompt.get('prompt_id')}: no target_frames_20fps and no length rule "
                f"for difficulty {difficulty!r} (rule: {self.length_by_difficulty})"
            )
        return int(self.length_by_difficulty[difficulty]), f"difficulty:{difficulty}"

    @staticmethod
    def _safe_prompt_name(prompt: str) -> str:
        # Mirrors MotionHiFlow gen_t2m.py.
        return prompt[:100].replace("/", "_").replace(" ", "_")

    @staticmethod
    def _safe_id(prompt: str, prompt_id: Optional[str]) -> str:
        if prompt_id:
            return re.sub(r"[^A-Za-z0-9_.-]", "_", str(prompt_id))
        # No ID given: derive a stable one from the prompt text.
        return "prompt_" + hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:10]

    def _call_output_root(self, prompt: str, prompt_id: Optional[str],
                          length: int, seed: int) -> Path:
        """One isolated output directory per (prompt_id, length, seed)."""
        tag = f"{self._safe_id(prompt, prompt_id)}_len{length}_seed{seed}"
        return self.repo_dir / "benchmark_runs" / "runner" / tag

    def _find_generated_joint_file(self, output_root: Path, prompt: str,
                                   length: int, repeat_index: int = 0) -> Path:
        """
        Locate the raw joint file inside this call's own output directory.

        Layout written by gen_t2m.py with output_dir=<output_root>:
            <output_root>/t2m_tmdit_16d/joints/<prompt>/sample0_r{k}_len{L}.npy
        """
        d = output_root / EXPERIMENT_NAME / "joints" / self._safe_prompt_name(prompt)

        if not d.exists():
            raise FileNotFoundError(
                f"MotionHiFlow output directory not found:\n{d}"
            )

        exact = d / f"sample0_r{int(repeat_index)}_len{int(length)}.npy"
        if exact.exists():
            return exact

        # Fall back to any plain joint file for this repeat, excluding IK and
        # feature files. The directory is fresh, so at most one should exist.
        pattern = re.compile(rf"^sample0_r{int(repeat_index)}_len\d+\.npy$")
        candidates = [p for p in d.glob("*.npy") if pattern.match(p.name)]

        if len(candidates) == 1:
            return candidates[0]

        found = sorted(p.name for p in d.iterdir())
        raise FileNotFoundError(
            "Could not uniquely identify the raw MotionHiFlow joint file in:\n"
            f"{d}\nContents: {found}"
        )

    def standardize_xyz(self, motion_or_path, return_metadata: bool = False):
        """
        Convert a raw MotionHiFlow joint array (or .npy path) to the benchmark
        convention: ground Y = 0, initial root XZ = (0, 0), +X = Right.

        MotionHiFlow gen_t2m.py saves the recovered ORIGINAL joints as
        [T, 22, 3]. No IK output is used for benchmark standardisation.
        """
        import numpy as np

        if isinstance(motion_or_path, (str, Path)):
            motion = np.load(motion_or_path)
        else:
            motion = np.asarray(motion_or_path)

        motion_std, metadata = standardise_motionhiflow(
            motion, mirror_x=self.mirror_x, n_joints=N_JOINTS
        )

        # Facing is estimated on the RAW motion (mirroring flips the angle sign).
        forward, angle_deg = estimate_hml3d_initial_forward(motion)
        if angle_deg is None:
            facing_status = "UNABLE_TO_CHECK"
        elif abs(angle_deg) <= self.facing_tolerance_deg:
            facing_status = "OK_+Z"
        else:
            facing_status = "CHECK_FACING"

        metadata.update({
            "Frames": int(len(motion_std)),
            "Facing_Angle_Deg": angle_deg,
            "Facing_Status": facing_status,
        })

        motion_std = motion_std.astype(np.float32, copy=False)
        if return_metadata:
            return motion_std, metadata
        return motion_std

    def generate(
        self,
        prompt: str,
        prompt_id: Optional[str] = None,
        motion_length: Optional[int] = None,
        repeat_index: int = 0,
        seed: Optional[int] = None,
        standardise: bool = True,
    ):
        """
        Generate one motion and return it as [T, 22, 3].

        Args:
            prompt        : benchmark prompt text.
            prompt_id     : official ID such as "C1-01"; used for the output
                            directory and recorded in last_metadata.
            motion_length : requested frames at 20 fps (100 / 150 / 196 by default);
                            MotionHiFlow rounds down to a multiple of 4 (150 -> 148).
            seed          : overrides the runner's seed for this call only.
            standardise   : True (default) returns benchmark-standard joints;
                            False returns the raw MotionHiFlow joints.

        Details of the call are available afterwards in `self.last_metadata`.
        """
        if not isinstance(prompt, str) or not prompt.strip():
            raise ValueError("prompt must be a non-empty string.")

        # gen_t2m.py strips each line of the prompt file and splits on "#",
        # so normalise whitespace here and reject "#" to keep the output
        # folder name predictable.
        prompt = " ".join(prompt.split())
        if "#" in prompt:
            raise ValueError("prompt must not contain '#' (MotionHiFlow prompt-file separator).")

        if motion_length is None:
            if self.default_motion_length is None:
                raise ValueError(
                    "motion_length is required (team rule: Easy 100, Medium 150, Hard 196)."
                )
            length = self.default_motion_length
        else:
            length = self._validate_length(motion_length)

        run_seed = self.seed if seed is None else int(seed)
        effective = self.effective_length(length)

        self.verify_checkpoints()

        # Fresh directory per call: a stale file from an earlier run can never
        # be mistaken for this call's output.
        output_root = self._call_output_root(prompt, prompt_id, length, run_seed)
        if output_root.exists():
            shutil.rmtree(output_root)
        output_root.mkdir(parents=True, exist_ok=True)

        # Pass the prompt through a text file, not `text_prompt=...`: Hydra
        # parses command-line overrides, so a comma turns the prompt into a
        # sweep list and parentheses or apostrophes are a syntax error.
        prompt_file = output_root / "prompt.txt"
        prompt_file.write_text(f"{prompt}#{length}\n", encoding="utf-8")

        cmd = [
            "bash",
            "run.sh",
            "gen",
            "tmdit",
            f"gpu_id={self.gpu_id}",
            f"seed={run_seed}",
            f"text_path={prompt_file}",
            "repeat_times=1",
            f"output_dir={output_root}",
        ]

        print("[MotionHiFlow] Generating...")
        print("[MotionHiFlow] Prompt:", prompt)
        if prompt_id is not None:
            print("[MotionHiFlow] Prompt ID:", prompt_id)
        print(f"[MotionHiFlow] Length: {length} requested -> {effective} frames | Seed: {run_seed}")

        self._run(cmd, cwd=self.repo_dir)

        joint_file = self._find_generated_joint_file(
            output_root=output_root,
            prompt=prompt,
            length=effective,
            repeat_index=repeat_index,
        )

        motion, std_meta = self.standardize_xyz(joint_file, return_metadata=True)

        if motion.shape[0] != effective:
            raise RuntimeError(
                f"Generated {motion.shape[0]} frames, expected {effective} "
                f"(requested {length}, prompt_id={prompt_id})."
            )

        if not standardise:
            import numpy as np
            motion = np.load(joint_file).astype(np.float32, copy=False)

        self.last_metadata = {
            "runner_version": RUNNER_VERSION,
            "prompt_id": prompt_id,
            "prompt": prompt,
            "motion_length": length,
            "effective_frames": effective,
            "seed": run_seed,
            "raw_joint_file": str(joint_file),
            "standardised": bool(standardise),
            "coordinate_system": (
                "benchmark_global_xyz" if standardise else "motionhiflow_raw"
            ),
            **std_meta,
        }

        print("[MotionHiFlow] Joint file:", joint_file)
        print(
            f"[MotionHiFlow] {'Standardised' if standardise else 'Raw'}: "
            f"{motion.shape} | facing={std_meta['Facing_Status']}"
        )

        return motion

    # ------------------------------------------------------------------
    # Batch export for the common benchmark notebook
    # ------------------------------------------------------------------

    @staticmethod
    def _load_benchmark_prompts(benchmark_path, split, prompt_ids):
        benchmark_path = Path(benchmark_path)
        definition = json.loads(benchmark_path.read_text(encoding="utf-8"))

        if "prompts" not in definition:
            raise KeyError(f"{benchmark_path} does not contain 'prompts'.")

        prompts = definition["prompts"]

        if split is not None:
            prompts = [p for p in prompts if p.get("set") == split]

        if prompt_ids is not None:
            wanted = list(prompt_ids)
            by_id = {p["prompt_id"]: p for p in prompts}
            unknown = [pid for pid in wanted if pid not in by_id]
            if unknown:
                raise KeyError(f"Prompt IDs not found in the selected split: {unknown}")
            prompts = [by_id[pid] for pid in wanted]

        if not prompts:
            raise RuntimeError(
                f"No prompts selected (split={split!r}, prompt_ids={prompt_ids!r})."
            )

        ids = [p["prompt_id"] for p in prompts]
        duplicates = sorted({x for x in ids if ids.count(x) > 1})
        if duplicates:
            raise RuntimeError(f"Duplicate Prompt IDs: {duplicates}")

        return definition, prompts

    def _run_generation_chunk(self, items, length, seed, chunk_dir, batch_size):
        """
        Generate one chunk of prompts that share a target length with a single
        run.sh call. `items` is a list of (prompt_id, normalised_text).

        Returns {prompt_id: raw_joint_file}.
        """
        if chunk_dir.exists():
            shutil.rmtree(chunk_dir)
        chunk_dir.mkdir(parents=True, exist_ok=True)

        # One "prompt#frames" line per entry. Line k produces sample{k}_... .
        prompt_file = chunk_dir / "prompts.txt"
        prompt_file.write_text(
            "".join(f"{text}#{length}\n" for _, text in items),
            encoding="utf-8",
        )
        (chunk_dir / "prompt_ids.json").write_text(
            json.dumps([pid for pid, _ in items], indent=2), encoding="utf-8"
        )

        cmd = [
            "bash", "run.sh", "gen", "tmdit",
            f"gpu_id={self.gpu_id}",
            f"seed={seed}",
            f"text_path={prompt_file}",
            "repeat_times=1",
            f"batch_size={int(batch_size)}",
            f"output_dir={chunk_dir}",
        ]
        self._run(cmd, cwd=self.repo_dir)

        joints_root = chunk_dir / EXPERIMENT_NAME / "joints"
        found = {}
        for k, (pid, text) in enumerate(items):
            folder = joints_root / self._safe_prompt_name(text)
            path = folder / f"sample{k}_r0_len{self.effective_length(length)}.npy"
            if not path.exists():
                contents = sorted(p.name for p in folder.iterdir()) if folder.exists() else "missing"
                raise FileNotFoundError(
                    f"Expected MotionHiFlow output for {pid} not found:\n{path}\n"
                    f"Folder contents: {contents}"
                )
            found[pid] = path
        return found

    def export_benchmark_package(
        self,
        benchmark_path,
        split: Optional[str] = "Pilot",
        out_dir="/content/benchmark_exports",
        model_name: str = "MotionHiFlow",
        seed: Optional[int] = None,
        prompt_ids: Optional[Iterable[str]] = None,
        batch_size: int = 16,
        chunk_size: int = 200,
        resume: bool = True,
        make_zip: bool = True,
        make_gifs: bool = True,
        gif_fps: int = 20,
    ) -> dict:
        """
        Generate every prompt of a benchmark split in batches and write the
        package the common benchmark notebook reads in STEP 3.5:

            <out_dir>/<model_name>/<prompt_id>.npy    standardised [T, 22, 3]
            <out_dir>/<model_name>/<prompt_id>.json   per-prompt metadata
            <out_dir>/<model_name>/manifest.json      run-level manifest
            <out_dir>/<model_name>_standardized_<split>.zip

        Lengths: target_frames_20fps when the definition gives it, otherwise the
        team rule by difficulty (self.length_by_difficulty; default Easy 100,
        Medium 150, Hard 196). MotionHiFlow rounds to a multiple of 4, so a
        requested 150 is generated as 148 frames; both numbers are recorded.

        Prompts are grouped by requested length and split into chunks of at
        most `chunk_size`; each chunk is one run.sh call, so the model is loaded
        once per chunk instead of once per prompt.

        Args:
            benchmark_path : benchmark definition JSON.
            split          : value of the prompts' "set" field ("Pilot", "Main");
                             None selects every prompt in the file.
            seed           : defaults to the runner's seed.
            prompt_ids     : optional subset, e.g. to regenerate a few prompts.
            batch_size     : gen_t2m.py batch size (it halves itself on OOM).
            chunk_size     : prompts per run.sh call; also the resume granularity.
            resume         : skip chunks whose .npy files already exist in the
                             package (with matching seed), so an interrupted
                             1000-prompt run continues where it stopped.
            make_gifs      : render every standardised motion with the original
                             render_standard_gif into <out_dir>/<model>_gifs/ and
                             zip them as <model>_gifs_<split>.zip (separate from
                             the evaluation ZIP). Existing GIFs are kept unless
                             their motion was regenerated in this run.

        Note on seeds: a batch is sampled from one RNG stream, so an individual
        motion depends on the batch it was generated in. Re-running with the
        same prompts, split, chunk_size and batch_size reproduces the package;
        it will not match single-prompt generate() calls with the same seed.
        """
        import numpy as np

        run_seed = self.seed if seed is None else int(seed)
        chunk_size = max(1, int(chunk_size))

        self.verify_checkpoints()

        definition, prompts = self._load_benchmark_prompts(
            benchmark_path, split, prompt_ids
        )

        # ---- validate and group by length (keeps benchmark order inside a group)
        groups: "OrderedDict[int, list]" = OrderedDict()
        prompt_by_id = {}
        length_source = {}
        for p in prompts:
            pid = p["prompt_id"]
            requested, source = self.length_for_prompt(p)
            length = self._validate_length(requested)
            length_source[pid] = source
            text = " ".join(str(p["text"]).split())
            if not text:
                raise ValueError(f"{pid}: empty prompt text")
            if "#" in text:
                raise ValueError(f"{pid}: prompt text must not contain '#'")
            groups.setdefault(length, []).append((pid, text))
            prompt_by_id[pid] = p

        out_dir = Path(out_dir)
        package_dir = out_dir / model_name
        package_dir.mkdir(parents=True, exist_ok=True)

        split_slug = (split or "all").lower()
        work_root = self.repo_dir / "benchmark_runs" / "export" / f"{split_slug}_seed{run_seed}"

        print(f"[MotionHiFlow] Export {len(prompts)} prompts | split={split} | seed={run_seed}")
        for length, items in sorted(groups.items()):
            print(f"[MotionHiFlow]   {length} requested -> {self.effective_length(length)} frames: "
                  f"{len(items)} prompts")

        records = {}
        regenerated = set()
        skipped_chunks = 0
        generated_chunks = 0

        for length, items in sorted(groups.items()):
            for start in range(0, len(items), chunk_size):
                chunk = items[start:start + chunk_size]
                chunk_tag = f"len{length}_chunk{start // chunk_size:03d}"

                if resume and self._chunk_complete(package_dir, chunk, run_seed, length):
                    skipped_chunks += 1
                    print(f"[MotionHiFlow] {chunk_tag}: already exported — skip.")
                    for pid, _ in chunk:
                        records[pid] = json.loads(
                            (package_dir / f"{pid}.json").read_text(encoding="utf-8")
                        )
                    continue

                print(f"[MotionHiFlow] {chunk_tag}: generating {len(chunk)} prompts...")
                raw_files = self._run_generation_chunk(
                    chunk, length, run_seed, work_root / chunk_tag, batch_size
                )
                generated_chunks += 1
                regenerated.update(pid for pid, _ in chunk)

                for pid, text in chunk:
                    raw_file = raw_files[pid]
                    motion, std_meta = self.standardize_xyz(raw_file, return_metadata=True)

                    if motion.shape[0] != self.effective_length(length):
                        raise RuntimeError(
                            f"{pid}: generated {motion.shape[0]} frames, expected "
                            f"{self.effective_length(length)} (requested {length})."
                        )

                    np.save(package_dir / f"{pid}.npy", motion)

                    p = prompt_by_id[pid]
                    record = {
                        "prompt_id": pid,
                        "model_name": model_name,
                        "prompt": text,
                        "set": p.get("set"),
                        "capability": p.get("capability"),
                        "difficulty": p.get("difficulty"),
                        "pair_id": p.get("pair_id"),
                        "motion_file": f"{pid}.npy",
                        "frames": int(motion.shape[0]),
                        "target_frames": int(length),
                        "length_source": length_source[pid],
                        "fps": 20,
                        "joint_count": N_JOINTS,
                        "seed": run_seed,
                        "coordinate_system": "benchmark_global_xyz",
                        "x_axis": "+X=Right, -X=Left",
                        "up_axis": "+Y",
                        "forward_axis": "+Z",
                        "units": "metres",
                        "ground_y": std_meta["New_Floor_Y"],
                        "initial_root_xz": [std_meta["New_Root_X"], std_meta["New_Root_Z"]],
                        "mirrored_x": std_meta["Mirrored_X"],
                        "facing_angle_deg": std_meta["Facing_Angle_Deg"],
                        "facing_status": std_meta["Facing_Status"],
                        "generation_chunk": chunk_tag,
                        "raw_joint_file": str(raw_file),
                    }
                    (package_dir / f"{pid}.json").write_text(
                        json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8"
                    )
                    records[pid] = record

        # ---- manifest (benchmark order)
        ordered = [records[p["prompt_id"]] for p in prompts]
        try:
            model_revision = subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=str(self.repo_dir),
                text=True, stderr=subprocess.DEVNULL,
            ).strip()
        except Exception:
            model_revision = None

        length_counts = {str(k): len(v) for k, v in sorted(groups.items())}
        facing_counts = {}
        for r in ordered:
            facing_counts[r["facing_status"]] = facing_counts.get(r["facing_status"], 0) + 1

        manifest = {
            "model_name": model_name,
            "runner_version": RUNNER_VERSION,
            "model_revision": model_revision,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "benchmark_file": Path(benchmark_path).name,
            "benchmark_sha256": hashlib.sha256(Path(benchmark_path).read_bytes()).hexdigest(),
            "benchmark_name": definition.get("benchmark_name"),
            "benchmark_schema_version": definition.get("schema_version"),
            "split": split,
            "seed": run_seed,
            "batch_size": int(batch_size),
            "chunk_size": chunk_size,
            "prompt_count": len(ordered),
            "length_counts": length_counts,
            "length_rule": {
                "by_difficulty": self.length_by_difficulty,
                "effective_frames": {str(k): self.effective_length(k) for k in groups},
                "note": "target_frames_20fps is used when present; otherwise the "
                        "difficulty rule. MotionHiFlow rounds to a multiple of 4.",
            },
            "facing_status_counts": facing_counts,
            "fps": 20,
            "joint_count": N_JOINTS,
            "units": "metres",
            "coordinate_system": {
                "x": "+X=Right, -X=Left",
                "y": "+Y=Up",
                "z": "+Z=Forward, -Z=Backward",
                "initial_root_xz": [0.0, 0.0],
                "ground_y": 0.0,
            },
            "python_version": platform.python_version(),
            "prompts": [
                {k: r[k] for k in ("prompt_id", "motion_file", "frames",
                                   "target_frames", "facing_status")}
                for r in ordered
            ],
        }
        (package_dir / "manifest.json").write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )

        # ---- zip: <model_name>/... at the archive root
        zip_path = None
        if make_zip:
            zip_path = out_dir / f"{model_name}_standardized_{split_slug}.zip"
            if zip_path.exists():
                zip_path.unlink()
            keep = {f"{r['prompt_id']}.npy" for r in ordered}
            keep |= {f"{r['prompt_id']}.json" for r in ordered}
            keep.add("manifest.json")
            with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
                for name in sorted(keep):
                    zf.write(package_dir / name, arcname=f"{model_name}/{name}")

        # ---- GIFs (original MotionHiFlow-pipeline renderer), separate ZIP
        gif_dir = gif_zip_path = None
        if make_gifs:
            gif_dir = out_dir / f"{model_name}_gifs"
            gif_dir.mkdir(parents=True, exist_ok=True)
            for i, r in enumerate(ordered, start=1):
                pid = r["prompt_id"]
                gif = gif_dir / f"{pid}.gif"
                if gif.exists() and pid not in regenerated:
                    continue
                print(f"[MotionHiFlow] GIF {i}/{len(ordered)}: {pid}")
                render_standard_gif(np.load(package_dir / f"{pid}.npy"), gif, pid, r["prompt"],
                                    fps=gif_fps, verbose=False)
            gif_zip_path = out_dir / f"{model_name}_gifs_{split_slug}.zip"
            if gif_zip_path.exists():
                gif_zip_path.unlink()
            with zipfile.ZipFile(gif_zip_path, "w") as zf:
                for r in ordered:
                    zf.write(gif_dir / f"{r['prompt_id']}.gif", arcname=f"{r['prompt_id']}.gif")

        print(
            f"[MotionHiFlow] Export finished: {len(ordered)} motions "
            f"({generated_chunks} chunk(s) generated, {skipped_chunks} resumed)."
        )
        print(f"[MotionHiFlow] Facing status: {facing_counts}")
        if facing_counts.get("CHECK_FACING"):
            print(
                "[MotionHiFlow] Note: some clips do not start facing +Z; "
                "see facing_status in the per-prompt .json files."
            )
        print(f"[MotionHiFlow] Package: {package_dir}")
        if zip_path:
            print(f"[MotionHiFlow] ZIP    : {zip_path}")
        if gif_zip_path:
            print(f"[MotionHiFlow] GIFs   : {gif_zip_path}")

        return {
            "package_dir": str(package_dir),
            "zip_path": str(zip_path) if zip_path else None,
            "gif_dir": str(gif_dir) if gif_dir else None,
            "gif_zip_path": str(gif_zip_path) if gif_zip_path else None,
            "manifest": manifest,
            "records": ordered,
        }

    @staticmethod
    def _chunk_complete(package_dir, chunk, seed, length) -> bool:
        """True when every prompt of the chunk already has a matching export."""
        for pid, text in chunk:
            npy = package_dir / f"{pid}.npy"
            meta = package_dir / f"{pid}.json"
            if not (npy.exists() and meta.exists()):
                return False
            try:
                record = json.loads(meta.read_text(encoding="utf-8"))
            except Exception:
                return False
            if (record.get("seed") != seed
                    or record.get("target_frames") != length
                    or record.get("prompt") != text):
                return False
        return True


def get_motionhiflow_runner(
    setup=True,
    install_dependencies=True,
    download_checkpoints=True,
    repo_dir="/content/MotionHiFlow",
    gpu_id=0,
    seed=1,
    default_motion_length=None,
    allowed_lengths=BENCHMARK_LENGTHS,
    mirror_x=True,
):
    runner = MotionHiFlowRunner(
        repo_dir=repo_dir,
        gpu_id=gpu_id,
        seed=seed,
        default_motion_length=default_motion_length,
        allowed_lengths=allowed_lengths,
        mirror_x=mirror_x,
    )

    if setup:
        runner.setup_environment(
            install_dependencies=install_dependencies,
            download_checkpoints=download_checkpoints,
        )

    return runner


## 3. Set up MotionHiFlow

Clones the repository, installs the verified dependency versions, applies the NumPy patches and
downloads the checkpoints, CLIP, evaluator and GloVe files. The first run takes a while; re-runs
skip finished steps. Generation runs in a separate process, so **no restart is needed**.

In [ ]:
# Fall back to the Step 1 defaults if Step 1 was skipped or the runtime restarted.
for _name, _value in {"SPLIT": "Pilot", "SEED": 1, "BATCH_SIZE": 16, "CHUNK_SIZE": 200, "MAKE_GIFS": True,
                      "USE_DRIVE": False,
                      "DRIVE_DIR": "/content/drive/MyDrive/motionhiflow_benchmark_exports"}.items():
    if _name not in globals():
        globals()[_name] = _value
        print(f"Note: {_name} not set (Step 1 not run) — using default {_value!r}")

import importlib, sys
sys.path.insert(0, "/content")

import motionhiflow_benchmark_runner as mhf
importlib.reload(mhf)   # pick up the file from Step 2 if this cell is re-run
print("Runner version:", mhf.RUNNER_VERSION)

runner = mhf.get_motionhiflow_runner(
    setup=True,
    install_dependencies=True,
    download_checkpoints=True,
    seed=SEED,
)

## 4. Upload the benchmark definition and check the lengths

In [ ]:
# Fall back to the Step 1 defaults if Step 1 was skipped or the runtime restarted.
for _name, _value in {"SPLIT": "Pilot", "SEED": 1, "BATCH_SIZE": 16, "CHUNK_SIZE": 200, "MAKE_GIFS": True,
                      "USE_DRIVE": False,
                      "DRIVE_DIR": "/content/drive/MyDrive/motionhiflow_benchmark_exports"}.items():
    if _name not in globals():
        globals()[_name] = _value
        print(f"Note: {_name} not set (Step 1 not run) — using default {_value!r}")

from pathlib import Path
import json

BENCHMARK_PATH = Path("/content/benchmark_definition.json")

if not BENCHMARK_PATH.exists():
    from google.colab import files
    print("Upload the benchmark definition JSON (e.g. pilot_benchmark_definition_model_independent_v1.2.json)")
    uploaded = files.upload()
    json_files = [n for n in uploaded if n.lower().endswith(".json")]
    if len(json_files) != 1:
        raise RuntimeError(f"Expected exactly one .json file, got {list(uploaded)}")
    Path(json_files[0]).replace(BENCHMARK_PATH)

definition = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
selected = [p for p in definition["prompts"] if SPLIT is None or p.get("set") == SPLIT]

# Length per prompt: target_frames_20fps if the definition has it, otherwise the
# team rule by difficulty (Easy 100, Medium 150 -> 148 generated, Hard 196).
counts = {}
for p in selected:
    requested, source = runner.length_for_prompt(p)
    key = (requested, runner.effective_length(requested), source.split(":")[0])
    counts[key] = counts.get(key, 0) + 1

print("Benchmark :", definition.get("benchmark_name"), "|", definition.get("schema_version"))
print("Split     :", SPLIT)
print("Prompts   :", len(selected))
for (requested, effective, source), n in sorted(counts.items()):
    print(f"  {n:>3} prompts: {requested} requested -> {effective} frames  ({source})")
if not selected:
    raise RuntimeError(f"No prompts with set == {SPLIT!r} in {BENCHMARK_PATH.name}")

## 5. Output folder

In [ ]:
# Fall back to the Step 1 defaults if Step 1 was skipped or the runtime restarted.
for _name, _value in {"SPLIT": "Pilot", "SEED": 1, "BATCH_SIZE": 16, "CHUNK_SIZE": 200, "MAKE_GIFS": True,
                      "USE_DRIVE": False,
                      "DRIVE_DIR": "/content/drive/MyDrive/motionhiflow_benchmark_exports"}.items():
    if _name not in globals():
        globals()[_name] = _value
        print(f"Note: {_name} not set (Step 1 not run) — using default {_value!r}")

from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(DRIVE_DIR)
else:
    OUT_DIR = Path("/content/benchmark_exports")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT_DIR)

## 6. Left/right sanity check (recommended)

Exports only C2-01 (*walks to the left*) and C2-02 (*walks to the right*) into a separate test
folder. Under the benchmark convention (+X = Right) C2-01 should move toward −X and C2-02 toward +X.

`ok=False` for both means the axis convention is wrong — stop and check. One failure alone can
simply be a model failure. `ok=None` means the clip barely moved sideways.

In [ ]:
# Fall back to the Step 1 defaults if Step 1 was skipped or the runtime restarted.
for _name, _value in {"SPLIT": "Pilot", "SEED": 1, "BATCH_SIZE": 16, "CHUNK_SIZE": 200, "MAKE_GIFS": True,
                      "USE_DRIVE": False,
                      "DRIVE_DIR": "/content/drive/MyDrive/motionhiflow_benchmark_exports"}.items():
    if _name not in globals():
        globals()[_name] = _value
        print(f"Note: {_name} not set (Step 1 not run) — using default {_value!r}")
missing = [n for n in ("runner", "mhf", "BENCHMARK_PATH", "selected") if n not in globals()]
if missing:
    raise RuntimeError(f"Run the earlier steps first — not defined: {missing}")

import numpy as np

probe_ids = [pid for pid in ("C2-01", "C2-02") if any(p["prompt_id"] == pid for p in selected)]

if not probe_ids:
    print("C2-01 / C2-02 are not in this split — skipping the check.")
else:
    probe = runner.export_benchmark_package(
        BENCHMARK_PATH, split=SPLIT, prompt_ids=probe_ids,
        out_dir="/content/benchmark_exports_probe", seed=SEED,
        batch_size=BATCH_SIZE, resume=False, make_zip=False, make_gifs=False,
    )
    expected = {"C2-01": "left", "C2-02": "right"}
    for r in probe["records"]:
        motion = np.load(Path(probe["package_dir"]) / r["motion_file"])
        ok, dx = mhf.check_left_right_convention(motion, expect=expected[r["prompt_id"]])
        print(f'{r["prompt_id"]} | expected={expected[r["prompt_id"]]:<5} | ok={ok} | '
              f'root dX={dx:+.3f} m | shape={motion.shape} | facing={r["facing_status"]}')

## 7. Export the whole split

Re-running this cell resumes: finished chunks are skipped. To redo specific prompts, delete their
`.npy` in the output folder and run the cell again.

In [ ]:
# Fall back to the Step 1 defaults if Step 1 was skipped or the runtime restarted.
for _name, _value in {"SPLIT": "Pilot", "SEED": 1, "BATCH_SIZE": 16, "CHUNK_SIZE": 200, "MAKE_GIFS": True,
                      "USE_DRIVE": False,
                      "DRIVE_DIR": "/content/drive/MyDrive/motionhiflow_benchmark_exports"}.items():
    if _name not in globals():
        globals()[_name] = _value
        print(f"Note: {_name} not set (Step 1 not run) — using default {_value!r}")
missing = [n for n in ("runner", "BENCHMARK_PATH", "OUT_DIR") if n not in globals()]
if missing:
    raise RuntimeError(f"Run the earlier steps first — not defined: {missing}")

report = runner.export_benchmark_package(
    BENCHMARK_PATH,
    split=SPLIT,
    out_dir=OUT_DIR,
    seed=SEED,
    batch_size=BATCH_SIZE,
    chunk_size=CHUNK_SIZE,
    resume=True,
    make_gifs=MAKE_GIFS,
)

manifest = report["manifest"]
print()
print("Prompts      :", manifest["prompt_count"])
print("Lengths      :", manifest["length_counts"])
print("Facing status:", manifest["facing_status_counts"])
print("ZIP          :", report["zip_path"])

## 8. Check and download the ZIPs (motions + GIFs)

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files
missing = [n for n in ("report", "manifest") if n not in globals()]
if missing:
    raise RuntimeError(f"Run the earlier steps first — not defined: {missing}")

zip_path = Path(report["zip_path"])
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()

n_npy = sum(n.endswith(".npy") for n in names)
bad = [n for n in names if not n.startswith("MotionHiFlow/")]
print(f"{zip_path.name}: {n_npy} motions, {zip_path.stat().st_size / 1e6:.1f} MB")
if bad or n_npy != manifest["prompt_count"]:
    raise RuntimeError(f"Unexpected ZIP contents: {n_npy} .npy, stray entries: {bad[:5]}")

files.download(str(zip_path))

if report.get("gif_zip_path"):
    gif_zip = Path(report["gif_zip_path"])
    print(f"{gif_zip.name}: {gif_zip.stat().st_size / 1e6:.1f} MB")
    files.download(str(gif_zip))

    # Preview one GIF inline
    from IPython.display import Image as _Image, display
    display(_Image(filename=str(Path(report["gif_dir"]) / f"{manifest['prompts'][0]['prompt_id']}.gif")))

## Next step

In `Benchmark_simple_version_English.ipynb`:

1. STEP 3.5 — upload `MotionHiFlow_standardized_<split>.zip`.
2. STEP 4 — set `SELECTED_MODEL = "MotionHiFlow"`.
3. Continue with the remaining steps unchanged.